In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
true_b = -1
true_k = 3
N = 100
# 数据生成
np.random.seed(42)
x = np.random.rand(N, 1)
epsilon = (.1 * np.random.randn(N, 1))
y = true_b + true_k * x + epsilon

In [ ]:
# 索引乱序
idx = np.arange(N)
np.random.shuffle(idx)

# 使用80个随机的数据用作训练
train_idx = idx[:int(N*.8)]
# 剩余的20个数据用作验证
val_idx = idx[int(N*.8):]

# 产生训练和验证数据集
x_train, y_train = x[train_idx], y[train_idx]
x_val, y_val = x[val_idx], y[val_idx]

In [ ]:
# 转换为张量
x_train_tensor = torch.as_tensor(x_train, dtype=torch.float32)
y_train_tensor = torch.as_tensor(y_train, dtype=torch.float32)
# 指定随机种子数方便比较
torch.manual_seed(23)


In [ ]:
# 定义模型
class MyLinearRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)  # 输入和输出的维度都是1

    def forward(self, x):
        return self.linear(x)

# 实例化模型
model = MyLinearRegression()


+ ### MyLinearRegression类从nn.Module继承，在初始化函数中，从nn.Linear类创建了一个linear对象。
+ ### Linear类的第一个参数是输入维度，对于本例而言输入维度为1，只有一个k值。第二个参数是输出维度，同样计算出的对应y值也只有1个，所以维度也是1。第三个参数是是否需要偏置，默认是True。所以实际有两个需要训练的参数，k和b。偏置b是可选的，通过出传递参数bias=False，可以取消偏置b。此处，这个默认值刚好符合需要，因此就不用写出来了。

In [ ]:
# 可以通过state_dict()来看到初始化的参数：
model_state_dict = model.state_dict()
model_state_dict['linear.weight'], model_state_dict['linear.bias']  


### 最后，需要注意的是虽然模型提供forward函数来预测值，但是不要直接调用forward，例如像这样：

### `model.forward(x) # 这样写不对`

### 而应该按照PyTorch的约定这样用：

### `model(x)`

> ### 现在来处理损失计算。不出所料，PyTorch提供了很多预先定义好的损失函数。只需要根据任务选择损失函数。由于是回归，我们使用均方误差(MSE)作为损失，因此需要PyTorch的nn.MSELoss类。

In [ ]:
# 定义损失函数
loss_fn = nn.MSELoss(reduction='mean')

+ ### 注意，nn.MSELoss()不是损失函数本身。nn.MSELoss是一个类。我们实例化了一个这个类的对象，将其称为loss_fn，用这个对象来计算损失。
+ ### 因此，可以给它传递一个预测值和一个标签，得到相应的损失值。
+ ### 此外，还可以指定要应用的计算方法，也就是说，如何对单个点的误差进行汇总？可以取它们的平均值(reduction="mean")，或者简单地把它们加起来(reduction="sum")。在示例中，使用典型的均值来计算MSE。如果使用sum计算，我们实际上是在计算SSE(sum of squared errors，误差平方和)。有了损失对象之后，可以给损失对象传递预测值和实际值来计算损失：

### `loss = loss_fn(yhat, y_train_tensor)`

> ### 需要注意的是，虽然loss_fn是一个对象，但看起来如同一个函数一样的使用了这个对象。这样的用法和前面模型的用法是一样的。也就是说实际调用了loss_fn对象中的forward函数计算的损失，nn.MESloss也是从Module类继承的！

### 调用的第一个参数是预测值，第二个是实际值（标签）。计算出损失之后，需要进一步计算梯度，此时只需要调用backward()函数：

### `loss.backward()`

In [ ]:
# 定义一个SGD优化器来更新参数
# 学习率
lr = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=lr)

+ ### 在7.2节，计算完梯度之后手动更新参数。这对于两个参数来说可能没问题，但如果参数很多呢？此时需要使用PyTorch的优化器，如SGD、RMSprop或Adam。
+ ### PyTorch有许多优化器，SGD是其中最基本的一个，而Adam是最受欢迎的一个。
+ ### 不同的优化器使用不同的机制来更新参数，但它们都是通过不同的路径实现相同的目标。
+ ### 和loss类似，在使用优化器之前我们需要先创建一个优化器。创建优化器的时候需要2个基本的参数，第一个是需要指出那些参数需要更新，第二个参数是学习率。

### 有了优化器之后，可以调用优化器的step()函数更新所有参数
### `optimizer.step()`
### `optimizer.zero_grad()`

> ### 注意到代码的第二行，在更新参数之后，把所有的梯度值置为了0。这样做的原因是PyTorch默认每次计算完梯度之后，会将当前梯度值和上一次的梯度值累加起来作为新的梯度值。这并不是我们需要的，我们计算每次的梯度，不需要累加。

In [ ]:
# 训练
# 定义训练轮数
epochs = 1000

for epoch in range(epochs):
    # 将模型设置为训练模式
    model.train()
    # step 1 计算模型的预测值
    yhat = model(x_train_tensor)

    # step 2 计算损失
    loss = loss_fn(yhat, y_train_tensor)
    
    # step 3 计算梯度
    loss.backward()

    # step 4 更新参数和清零梯度
    optimizer.step()
    optimizer.zero_grad()

# 输出结果
print(model.state_dict())